# Evaluation Pipeline — Reference-Based Comparison

Compare synthesised audio (DDSP and baseline) against a **solo violin reference set**.

1. **Build reference** — extract and concatenate per-frame timbre features from all violin recordings.
2. **Strategy 1 (Distribution)** — compare concatenated folder-level distributions to the reference.
3. **Strategy 2 (Pairwise)** — compare each individual file to the reference distribution.
4. **Visualizations** — bar charts, box plots, heatmaps, method comparison.

In [1]:
import logging
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


In [ ]:
from evaluation.timbre_metrics import TimbreMetrics
from evaluation.loss import Loss
from evaluation.experiment_pipeline import collect_wav_files
from visualize import (
    plot_distribution_comparison,
    plot_loss_by_group,
    plot_loss_boxplot,
    plot_loss_heatmap,
    plot_method_comparison,
)

import numpy as np
import pandas as pd

## Paths

In [ ]:
# --- Input / output directories ---
REFERENCE_DIR  = PROJECT_ROOT / "data" / "raw" / "solo_violin"  # adjust to your reference path
DDSP_DIR       = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_transfered"
BASELINE_DIR   = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_baseline"

# --- Feature extraction config ---
FEATURE_TYPES  = ["spectral"]  # choose from: "spectral", "level", "harmonic"
CUSTOM_METRICS = None  # e.g. ["spectral_centroid", "spectral_spread", "spectral_flatness"]

# --- Result CSVs ---
RESULTS_DIR    = PROJECT_ROOT / "artifacts" / "evaluation"

print(f"Reference dir: {REFERENCE_DIR}  (exists: {REFERENCE_DIR.exists()})")
print(f"DDSP dir:      {DDSP_DIR}  (exists: {DDSP_DIR.exists()})")
print(f"Baseline dir:  {BASELINE_DIR}  (exists: {BASELINE_DIR.exists()})")
print(f"Feature types: {FEATURE_TYPES}")
print(f"Custom metrics: {CUSTOM_METRICS if CUSTOM_METRICS is not None else 'ALL (default for selected feature types)'}")
print(f"Results dir:   {RESULTS_DIR}")

Reference dir: /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/solo_violin  (exists: True)
DDSP dir:      /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_transfered  (exists: True)
Baseline dir:  /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline  (exists: True)
Results dir:   /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/evaluation


## 1. Extract Reference Features

Build the reference timbre distribution using `TimbreMetrics.extract_from_dir`,
which recursively extracts per-frame features from every WAV file and
concatenates them into a single array per feature.

In [ ]:
SR = 16000
tm = TimbreMetrics(sample_rate=SR)
loss = Loss()

# --- Reference ---
ref_features = tm.extract_from_dir(
    REFERENCE_DIR,
    metrics=CUSTOM_METRICS,
    feature_types=FEATURE_TYPES,
)
feat_keys = sorted(ref_features.keys())
ref_2d = np.column_stack([ref_features[k] for k in feat_keys])

print(f"Reference: {ref_2d.shape[0]} frames, {ref_2d.shape[1]} features")
print(f"Feature keys: {feat_keys}")

## 2. Strategy 1 — Distribution-Level Comparison

Compare the concatenated folder-level feature distribution of each method
against the reference.  Sub-sample to 10 000 frames for computational
efficiency (MMD is O(n^2)).

In [ ]:
SYNTHESIZED_DIRS = {"ddsp": DDSP_DIR, "baseline": BASELINE_DIR}

dist_rows = []
synth_features_cache = {}  # reused by pairwise step

# Step 1: Loop through each synthesis method and its output folder.
for method_name, synth_dir in SYNTHESIZED_DIRS.items():
    # Step 2: Extract and concatenate per-frame timbre features from all files.
    synth_features = tm.extract_from_dir(synth_dir)
    synth_features_cache[method_name] = synth_features

    # Step 3: Build a [n_frames, n_features] matrix in the same feature order as reference.
    synth_2d = np.column_stack([synth_features[k] for k in feat_keys])

    # Step 4: Use full reference/synth feature matrices (no frame capping).
    ref_sub = ref_2d
    synth_sub = synth_2d

    # Step 5: Compute distribution distances on the full distributions.
    dist_result = loss.evaluate(synth_sub, ref_sub)

    # Step 6: Save summary stats for this method.
    dist_rows.append({
        "method": method_name,
        "n_files": len(collect_wav_files(synth_dir)),
        "total_frames": synth_2d.shape[0],
        **dist_result,
    })
    print(f"{method_name}: mmd={dist_result['mmd']:.6f}, wasserstein={dist_result['wasserstein']:.6f}")

# Step 7: Convert results to a table and plot method-level comparisons.
df_distribution = pd.DataFrame(dist_rows)
display(df_distribution)

plot_distribution_comparison(df_distribution, "mmd")
plot_distribution_comparison(df_distribution, "wasserstein")

## 3. Strategy 2 — Pairwise File-vs-Reference

Compare each individual synthesised file's feature distribution against the
full reference distribution.

In [ ]:
from tqdm.auto import tqdm

pair_rows = []

# Step 1: Iterate over each synthesis method/output folder.
for method_name, synth_dir in SYNTHESIZED_DIRS.items():
    # Step 2: Collect all WAV files generated by this method.
    synth_dir_path = Path(synth_dir)
    wav_files = collect_wav_files(synth_dir_path)

    # Step 3: Compare each synthesized file to the reference distribution.
    for wf in tqdm(wav_files, desc=f"Pairwise [{method_name}]"):
        rel = str(wf.relative_to(synth_dir_path))
        try:
            # Step 4: Extract per-frame feature series for one file.
            series = tm.extract_series_from_file(wf)
            if not series:
                continue

            # Step 5: Keep only features present in both file and reference.
            available = sorted(set(series.keys()) & set(feat_keys))
            if not available:
                continue

            # Step 6: Align file and reference matrices on the same feature columns.
            col_idx = [feat_keys.index(k) for k in available]
            file_2d = np.column_stack([series[k] for k in available])
            ref_subset = ref_2d[:, col_idx]

            # Step 7: Compute distances for this file and save one output row.
            distances = loss.evaluate(file_2d, ref_subset)
            pair_rows.append({"method": method_name, "file": rel, **distances})
        except Exception:
            # Step 8: Skip files that fail extraction/evaluation and continue.
            continue

# Step 9: Build the final pairwise result table for summaries and plots.
df_pairwise = pd.DataFrame(pair_rows)
print(f"Pairwise results: {len(df_pairwise)} rows")
df_pairwise.head()

### Pairwise Summary by Method

In [ ]:
display(df_pairwise.groupby("method")[["mmd", "wasserstein"]].describe().round(4))

plot_loss_boxplot(df_pairwise, "method", "mmd")
plot_loss_boxplot(df_pairwise, "method", "wasserstein")

## 4. Method Comparison

In [9]:
plot_loss_by_group(df_pairwise, "method", "mmd")

In [10]:
plot_loss_by_group(df_pairwise, "method", "wasserstein")

## 5. Export

Save summary tables and figures.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# CSVs
df_distribution.to_csv(RESULTS_DIR / "distribution_summary.csv", index=False)
df_pairwise.to_csv(RESULTS_DIR / "evaluation_pairwise.csv", index=False)

pairwise_summary = df_pairwise.groupby("method")[["mmd", "wasserstein"]].agg(["mean", "std", "median"]).round(4)
pairwise_summary.to_csv(RESULTS_DIR / "pairwise_summary_by_method.csv")

# Figures
plot_distribution_comparison(df_distribution, "mmd", save_path=str(FIGURES_DIR / "distribution_mmd.png"))
plot_distribution_comparison(df_distribution, "wasserstein", save_path=str(FIGURES_DIR / "distribution_wasserstein.png"))
plot_loss_boxplot(df_pairwise, "method", "mmd", save_path=str(FIGURES_DIR / "pairwise_mmd_boxplot.png"))
plot_loss_boxplot(df_pairwise, "method", "wasserstein", save_path=str(FIGURES_DIR / "pairwise_wasserstein_boxplot.png"))
plot_loss_by_group(df_pairwise, "method", "mmd", save_path=str(FIGURES_DIR / "pairwise_mmd_by_method.png"))
plot_loss_by_group(df_pairwise, "method", "wasserstein", save_path=str(FIGURES_DIR / "pairwise_wasserstein_by_method.png"))

print(f"Exported to {RESULTS_DIR}")